# YOLO11-Seg: Dataset Processing and Training Notebook

This notebook converts the exported AirSim HDF5 dataset into YOLO segmentation format and trains a single-class `drone` model.

## 1. Environment Setup

In [ ]:
# If needed, uncomment the next line.
# !pip install -q ultralytics h5py opencv-python matplotlib numpy

import json
from pathlib import Path

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

print('Imports loaded.')

## 2. Paths and Training Config

In [ ]:
CONFIG = {
    'raw_dataset_root': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/dataset_airsim',
    'processed_root': r'E:/Programs/AirSim/Cosys-AirSim/PythonClient/YOLO-3D/yolo11_seg_dataset',
    'class_name': 'drone',
    'class_id': 0,
    'min_area_px': 25,
    'poly_eps_ratio': 0.002,
    'model': 'yolo11n-seg.pt',
    'imgsz': 960,
    'epochs': 100,
    'batch': 8,
    'device': 0,
    'workers': 4,
    'project': r'E:/Programs/AirSim/Cosys-AirSim/runs/segment',
    'run_name': 'yolo11n_seg_airsim_drone',
}

RAW_ROOT = Path(CONFIG['raw_dataset_root'])
OUT_ROOT = Path(CONFIG['processed_root'])
for split in ('train', 'val', 'test'):
    (OUT_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))

## 3. Convert HDF5 Export to YOLO Segmentation Format

In [ ]:
def collect_samples(raw_root):
    samples = []
    for split in ('train', 'val', 'test'):
        meta_dir = Path(raw_root) / 'metadata' / split
        if not meta_dir.exists():
            continue
        for meta_path in sorted(meta_dir.glob('*.json')):
            meta = json.loads(meta_path.read_text(encoding='utf-8'))
            h5_path = Path(raw_root) / meta['h5_path']
            if h5_path.exists():
                samples.append((split, meta_path, h5_path, meta))
    return samples

def contour_to_line(contour, width, height, class_id):
    contour = contour.reshape(-1, 2)
    if len(contour) < 3:
        return None
    coords = []
    for x, y in contour:
        coords.extend([float(x) / width, float(y) / height])
    return str(class_id) + ' ' + ' '.join(f'{v:.6f}' for v in coords)

def mask_to_lines(mask, width, height, class_id, min_area_px, poly_eps_ratio):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for contour in contours:
        if cv2.contourArea(contour) < min_area_px:
            continue
        epsilon = poly_eps_ratio * cv2.arcLength(contour, True)
        contour = cv2.approxPolyDP(contour, epsilon, True)
        line = contour_to_line(contour, width, height, class_id)
        if line is not None:
            lines.append(line)
    return lines

def build_dataset(config):
    samples = collect_samples(config['raw_dataset_root'])
    counts = {split: 0 for split in ('train', 'val', 'test')}

    for split, meta_path, h5_path, meta in tqdm(samples):
        with h5py.File(h5_path, 'r') as h5f:
            rgb = h5f['rgb_scene'][:]
            if 'segmentation_instance_id' not in h5f:
                continue
            seg = h5f['segmentation_instance_id'][:]

        frame_name = h5_path.stem
        image_path = OUT_ROOT / 'images' / split / f'{frame_name}.png'
        label_path = OUT_ROOT / 'labels' / split / f'{frame_name}.txt'
        cv2.imwrite(str(image_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

        objects = meta.get('objects', {})
        instance_ids = [
            int(obj['instance_id'])
            for obj in objects.get('spawned_objects', [])
            if obj.get('instance_id') not in (None, 0)
        ]
        if not instance_ids:
            instance_ids = [
                int(det['mask']['instance_id'])
                for det in objects.get('gt_detections', [])
                if det.get('mask', {}).get('instance_id') not in (None, 0)
            ]

        lines = []
        height, width = seg.shape[:2]
        for instance_id in sorted(set(instance_ids)):
            lines.extend(
                mask_to_lines(
                    seg == instance_id,
                    width,
                    height,
                    config['class_id'],
                    config['min_area_px'],
                    config['poly_eps_ratio'],
                )
            )

        label_path.write_text('\n'.join(lines), encoding='utf-8')
        counts[split] += 1

    yaml_path = OUT_ROOT / 'dataset.yaml'
    yaml_path.write_text(
        '\n'.join([
            f'path: {OUT_ROOT.as_posix()}',
            'train: images/train',
            'val: images/val',
            'test: images/test',
            'names:',
            f"  0: {config['class_name']}",
        ]),
        encoding='utf-8',
    )
    return counts, yaml_path

counts, dataset_yaml = build_dataset(CONFIG)
print('Dataset counts:', counts)
print('Dataset yaml:', dataset_yaml)

## 4. Quick Sanity Check

In [ ]:
sample_images = sorted((OUT_ROOT / 'images' / 'train').glob('*.png'))
sample_labels = sorted((OUT_ROOT / 'labels' / 'train').glob('*.txt'))
print('train images:', len(sample_images))
print('train labels:', len(sample_labels))

if sample_images:
    image = cv2.cvtColor(cv2.imread(str(sample_images[0])), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(sample_images[0].name)
    plt.show()

if sample_labels:
    print(sample_labels[0].read_text(encoding='utf-8')[:1000])

## 5. Train YOLO11-Seg

In [ ]:
from ultralytics import YOLO

model = YOLO(CONFIG['model'])
results = model.train(
    data=str(dataset_yaml),
    imgsz=CONFIG['imgsz'],
    epochs=CONFIG['epochs'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    workers=CONFIG['workers'],
    project=CONFIG['project'],
    name=CONFIG['run_name'],
)
results

## 6. Validate or Resume

In [ ]:
# Example:
# best_model = YOLO(Path(CONFIG['project']) / CONFIG['run_name'] / 'weights' / 'best.pt')
# metrics = best_model.val(data=str(dataset_yaml), split='val', device=CONFIG['device'])
# metrics